# Breaking the Blend - template for a new galaxy

A clean version of the automated Breaking the Blend workflow (Veenema et al. 2026) **with no galaxy-specific
settings**. Use it for any NIRSpec and/or MIRI/MRS (or other IFS) data: fill in the cells marked **EDIT**,
run, inspect, and refine. Defaults are generic starting points, not tuned values.

| Section | What happens |
|---|---|
| 0 | **EDIT** - name, data folder, redshift, run and figure switches |
| 1 | Find & load cubes, check the AGN position |
| 2 | **EDIT** - emission lines and (optional) per-line settings |
| 3 | **Step 1** - line maps and the dispersion of each component |
| 4 | **EDIT** - **Step 2** - kinematic models: which parameters are fixed / fitted, which lines trace what |
| 5 | **EDIT** - **Step 3** - decomposition: where the model parameters and dispersions come from |
| 6 | **Step 4** - stacked AGN outflow fraction map |
| 7 | **Step 5** - AGN- and star-formation-dominated spectra |
| 8 | **Corrected spectra** - extinction ($\tau_{9.8}$ map from H$_2$), then the PSF (AGN point source), corrected per spaxel and band, then stitched -> AGN-psf-corrected, AGN-outflow-corrected and SF-corrected spectra |

Section 8 corrects the Step 5 spectra spaxel by spaxel and band by band - first for extinction, with each spaxel's own
$\tau_{9.8}$ from the H$_2$ S(1)-S(7) rotation diagram, then for the AGN light spread by the PSF - and then stitches the bands:
AGN-psf-corrected, AGN-outflow-corrected and SF-corrected spectra, and a $\tau_{9.8}$ map on the grid of the outflow
fraction map. Its H$_2$ and PSF settings are generic; check the figures it shows.

Every slow step is saved in `RESULTS_DIR/GALAXY/` and re-loaded next time (recomputed automatically when its settings
change). Work through the sections in order the first time and look at every figure - the checks are what make the
result trustworthy. See `01_NGC5728.ipynb` for a fully worked example.

## 0. Settings - EDIT

In [ ]:
# =====================================================================
# 0. ANALYSIS SETTINGS
# =====================================================================
GALAXY       = "MyGalaxy"              # analysis name -> results are saved in RESULTS_DIR/GALAXY/
DATA_DIR     = "path/to/cubes"         # folder containing the *_s3d.fits(.gz) cubes (searched recursively)
RESULTS_DIR  = "results"
Z            = None                    # redshift  <-- REQUIRED
DISTANCE_MPC = None                    # optional distance [Mpc] for the physical scale (None = from Z)

# Re-run switches: False -> load saved results if they exist (and their settings are unchanged)
RERUN_LINE_FITS     = False
RERUN_KINEMATICS    = False
RERUN_DECOMPOSITION = False

# Kinematic fitting. "least_squares" is fast for a first pass; switch to "emcee" for final results
# (posteriors / uncertainties), e.g. dict(method="emcee", nwalkers=32, nsteps=20000, burnin=5000, sigma_v="residual")
MCMC = dict(method="least_squares", n_starts=20, sigma_v="residual")

# Figure output: "png", "pdf", "both" or "none"  (saved in RESULTS_DIR/GALAXY/figures/)
SAVE_FIGURES = "both"
FIGURE_DPI   = 200

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
try:
    import breaking_the_blend as btb
except ModuleNotFoundError:   # not installed (pip install -e .): use the copy in this repository
    import sys, os
    sys.path.insert(0, os.path.abspath(".."))
    import breaking_the_blend as btb
from breaking_the_blend import plotting as bplt     # also sets the publication plotting style

if Z is None:
    raise ValueError("Set the redshift Z in Section 0")
proj = btb.Project(GALAXY, data_dir=DATA_DIR, z=Z, distance_mpc=DISTANCE_MPC, results_dir=RESULTS_DIR,
                   save_figures=SAVE_FIGURES, figure_dpi=FIGURE_DPI)

## 1. Data: find the cubes and locate the AGN

All `*s3d.fits*` files under `DATA_DIR` are found and named by band (`ch3-short`, `g395h-f290lp`, ...). Use
`exclude=[...]` to skip files (e.g. background cubes) or `proj.set_cube_files({name: path})` to choose them by hand.

The AGN position of each cube is its (centroided) continuum peak. **Check every cross** in the overview figure; if
one is off the nucleus (e.g. a bright star-forming knot, or a weak AGN), give its pixel position in `AGN_OVERRIDES`.

In [ ]:
files = proj.find_cubes(exclude=["*bkg*"])

In [ ]:
AGN_OVERRIDES = {}            # EDIT if needed: {"cube name": (x, y)} in pixels
PIXEL_SCALE_OVERRIDES = {}    # normally empty: the header pixel size is used

LOAD_ERR = True               # also read the ERR extensions (spectrum errors in Step 5; doubles cube memory)

cubes = proj.load_cubes(agn_overrides=AGN_OVERRIDES, pixel_scale_overrides=PIXEL_SCALE_OVERRIDES, load_err=LOAD_ERR)
fig = bplt.plot_cubes_overview(cubes);  proj.savefig(fig, "cubes_overview")

## 2. Emission lines - EDIT

Start from a broad list; lines not covered by any loaded cube are skipped automatically. After Step 1, **remove lines
that are weak or blended** in your data. Per-line settings (`LINE_SETTINGS`) are only needed where the defaults
fail - look at the spaxel fits in Step 1 first, or test settings quickly with `03_check_line_settings.ipynb`. Useful settings: `line_region` (rest $\mu$m range of the line;
widen it for broad outflows), `fit_window`, `mask_percentile`, `p0_sigma_kms`, `p0_dv_kms`, `p0_amp_frac`, `cube`.
All options: `help(btb.LineSpec)`; the catalogue: `btb.LINE_CATALOGUE`.

In [ ]:
LINES = ["FeII", "ArII", "NeII", "SIII", "ArIII", "SIV", "NeIII", "OIV", "ArVI", "MgIV", "NeV", "NeVI"]

COMMON_SETTINGS = dict(mask_percentile=40)          # applied to every line (40 -> brightest 60% of spaxels)
LINE_SETTINGS = {
    # [Cl II] 14.368 um lies ~1000 km/s from [Ne V] 14.32 um: keep it out of the [Ne V] window
    "NeV": dict(line_region=(14.295, 14.345), fit_window=(14.265, 14.350)),
}
# Example of a full set of per-line settings (those used for NGC 5728 in notebook 01). Uncomment and adapt
# lines as needed - check them first with 03_check_line_settings.ipynb.
# LINE_SETTINGS = {
#     "FeII":  dict(cube="ch1-short",  line_region=(5.320, 5.355),  mask_percentile=55,
#                   p0_amp_frac=(0.67, 0.33), p0_dv_kms=(-2.8, 1.1),  p0_sigma_kms=(112, 225)),
#     "ArII":  dict(cube="ch1-long",   line_region=(6.975, 7.000),  mask_percentile=50,
#                   p0_amp_frac=(0.77, 0.50), p0_dv_kms=(-21.5, 4.3), p0_sigma_kms=(86, 215)),
#     "NeII":  dict(cube="ch3-short",  line_region=(12.800, 12.830), mask_percentile=70,
#                   p0_amp_frac=(0.50, 0.33), p0_dv_kms=(-23.4, 2.3), p0_sigma_kms=(105, 164)),
#     "SIII":  dict(cube="ch4-short",  line_region=(18.650, 18.780), mask_percentile=55,
#                   p0_amp_frac=(0.71, 0.40), p0_dv_kms=(-2.4, 3.2),  p0_sigma_kms=(112, 152)),
#     "ArIII": dict(cube="ch2-medium", line_region=(8.980, 9.015),  mask_percentile=55,
#                   p0_amp_frac=(0.71, 0.43), p0_dv_kms=(-1.7, 1.7),  p0_sigma_kms=(100, 200)),
#     "SIV":   dict(cube="ch2-long",   line_region=(10.485, 10.535), mask_percentile=55,
#                   p0_amp_frac=(0.67, 0.33), p0_dv_kms=(0.0, 28.5),  p0_sigma_kms=(86, 257)),
#     "NeIII": dict(cube="ch3-long",   line_region=(15.520, 15.600), mask_percentile=60,
#                   p0_amp_frac=(0.77, 0.33), p0_dv_kms=(-3.9, 3.9),  p0_sigma_kms=(106, 173)),
#     "OIV":   dict(cube="ch4-long",   line_region=(25.830, 25.970), mask_percentile=55,
#                   p0_amp_frac=(0.67, 0.33), p0_dv_kms=(0.0, 1.2),   p0_sigma_kms=(81, 127)),
#     "ArVI":  dict(cube="g395h-f290lp", line_region=(4.524, 4.537), mask_percentile=55,
#                   p0_amp_frac=(0.33, 0.50), p0_dv_kms=(-3.3, 3.3),  p0_sigma_kms=(66, 165)),
#     "MgIV":  dict(cube="g395h-f290lp", line_region=(4.481, 4.492), mask_percentile=55,
#                   p0_amp_frac=(0.33, 0.67), p0_dv_kms=(-0.7, 0.1),  p0_sigma_kms=(134, 267)),
#     "NeV":   dict(cube="ch3-medium", line_region=(14.295, 14.345), fit_window=(14.265, 14.350),  # avoid [Cl II]
#                   mask_percentile=50),
#     "NeVI":  dict(cube="ch2-short",  line_region=(7.645, 7.670),  mask_percentile=50,
#                   p0_amp_frac=(0.33, 0.67), p0_dv_kms=(-0.4, 0.0),  p0_sigma_kms=(78, 176)),
# }
# Note: `cube` names must match your loaded cubes, and settings tuned to one galaxy's line widths
# (line_region, p0_sigma_kms) may not suit another.

lines = btb.get_lines(LINES)
lines = {n: l.update(**{**COMMON_SETTINGS, **LINE_SETTINGS.get(n, {})}) for n, l in lines.items()}
lines = btb.assign_cubes(lines, cubes, drop_missing=True)
proj.save_config(lines=lines)

## 3. Step 1 - measure every line in every spaxel

Continuum subtraction, line flux, masking, single- and double-Gaussian fits and moment maps for every line.
The defaults are conservative: bounded fits (`fitter="trf"`), unweighted (`weighting="none"`; use `"err"` if your cubes
have reliable ERR extensions), started at each spaxel's moment-1 velocity, and the flux percentile computed inside the
field of view.

In [ ]:
FIT_OPTIONS = dict(fitter="trf", weighting="none", p0_center="moment", percentile_over="fov", dilate=2)

line_maps = proj.checkpoint("step1_line_maps", btb.fit_lines, cubes, lines, **FIT_OPTIONS,
                            rerun=RERUN_LINE_FITS, settings=dict(lines=lines, **FIT_OPTIONS))

**Check the fits.** Change `CHECK_LINE` to look at each line in turn. If the double-Gaussian fits look wrong, adjust
that line in `LINE_SETTINGS` (Section 2); it is re-fitted automatically.

In [ ]:
CHECK_LINE = list(line_maps)[len(line_maps) // 2]       # EDIT to inspect other lines
fig = bplt.plot_spaxel_fits(line_maps[CHECK_LINE], n=6);  proj.savefig(fig, f"spaxel_fits_{CHECK_LINE}")

In [ ]:
fig = bplt.plot_line_maps(line_maps, "flux");      proj.savefig(fig, "flux_maps")
fig = bplt.plot_line_maps(line_maps, "velocity");  proj.savefig(fig, "velocity_maps")
fig = bplt.plot_line_maps(line_maps, "sigma");     proj.savefig(fig, "single_gaussian_dispersion_maps")

Use these maps (ordered by IP) to decide, for Section 4, **which lines are dominated by the disc, which by the outflow,
and which show both**. Note the velocity colour range (`vlim=(-v, v)`) can be changed in every map function.

In [ ]:
fig = bplt.plot_line_quantities(line_maps, ("v_n", "v_b"));          proj.savefig(fig, "double_gaussian_velocity_maps")
fig = bplt.plot_line_quantities(line_maps, ("sigma_n", "sigma_b"));  proj.savefig(fig, "double_gaussian_dispersion_maps")
fig = bplt.plot_line_quantities(line_maps, ("mom0", "mom1", "mom2")); proj.savefig(fig, "moment_maps")

### Dispersion of each component

With $\sigma_{\rm outflow} > \sigma_{\rm disc}$ enforced per spaxel, the medians are the default fixed dispersions for
Step 3. For lines with a single component the narrow/broad split is meaningless - check the histograms, and use
`SIGMA_OVERRIDES = {"line": (sigma_disc, sigma_outflow)}` where needed.

In [ ]:
SIGMA_OVERRIDES = {}

sigmas_measured = btb.dispersion_medians(line_maps, sigma_range=(15, 600), overrides=SIGMA_OVERRIDES)
fig = bplt.plot_dispersion_histograms(line_maps, sigmas_measured, xlim=None)
proj.savefig(fig, "dispersion_histograms")
proj.save_config(sigmas_measured=sigmas_measured)

## 4. Step 2 - kinematic models - EDIT

Built-in models (offsets from the AGN in **pc**, angles in degrees, PA **east of north**):

* **thin rotating disc** - `v_disc`, `R_disc`, `inc_disc`, `pa_disc` (PA of the *receding* side);
* **hollow bicone** - `v_out`, `R_out`, `inc_out`, `pa_out` (PA of the *redshifted* cone), `theta_in`, `theta_out`;
* **combined** - $v = \alpha v_{\rm disc} + (1-\alpha) v_{\rm outflow}$, plus `v_sys`.

In `MODEL_PARAMS` a **number fixes** a parameter and a **tuple `(lo, hi)` fits** it with a uniform prior. Any
combination works. **If the inclinations and position angles are known (literature, other tracers), fix them** - a
partial field of view rarely constrains the geometry well. Below everything is free with broad priors as a starting
point. If a fitted model looks mirrored, the PA is 180 deg off.

Keep the `(lo, hi)` ranges **wide** unless you have a reason to restrict them. A fit that ends on the edge of its range
is reported as `<-- at prior limit`: that value is set by the prior rather than the data, so widen that range (or fix
the parameter deliberately). Least-squares errors are approximate (and zero for the opening angles, whose effect is
not smooth); use `method="emcee"` for meaningful uncertainties.

In [ ]:
MODEL_PARAMS = {
    "v_sys":     0.0,           # km/s  (0 if the redshift is right; e.g. (-100, 100) to fit)
    # ---- disc ---------------------------------------------------------------------------------
    "inc_disc":  (10, 80),      # deg   EDIT: fix to a known value if available, e.g. 45.0
    "pa_disc":   (0, 360),      # deg E of N (receding side)
    "v_disc":    (30, 600),     # km/s
    "R_disc":    (20, 2000),    # pc
    # ---- outflow -----------------------------------------------------------------------------
    "inc_out":   (10, 80),      # deg
    "pa_out":    (0, 360),      # deg E of N (redshifted cone)
    "v_out":     (50, 2000),    # km/s
    "R_out":     (50, 3000),    # pc
    "theta_in":  (0, 60),       # deg
    "theta_out": (20, 120),     # deg
    # ---- combined -----------------------------------------------------------------------------
    "alpha":     (0, 1),
}

**Line groups - EDIT after looking at the Step 1 maps.** The initial guess below uses the ionisation potential:
low-IP lines (< 25 eV) for the disc, high-IP lines (> 55 eV) for the outflow, and every line below 90 eV for the
decomposition. Lines with **no** disc component (check the narrow-component maps) should be left out of
`DECOMPOSE_LINES` - they still constrain the outflow.

In [ ]:
DISC_LINES      = [n for n, l in lines.items() if l.ip < 25]     # EDIT
OUTFLOW_LINES   = [n for n, l in lines.items() if l.ip > 55]     # EDIT
DECOMPOSE_LINES = [n for n, l in lines.items() if l.ip < 90]     # EDIT
print("disc:", DISC_LINES, "\noutflow:", OUTFLOW_LINES, "\ndecompose:", DECOMPOSE_LINES)

RUN_KINEMATIC_FITS = True
# For the geometry schematics only: which side of the disc minor axis is nearer to us
# (+1: the side at PA_disc + 90 deg, -1: PA_disc - 90 deg) - set from dust lanes if known.
DISC_NEAR_SIDE = +1
# Geometry fitted in the single-component fits (4.1, 4.2) is averaged and FIXED in the combined fits (4.3),
# which otherwise have too much freedom. Set False to keep it free there too.
FIX_GEOMETRY_IN_COMBINED = True

In [ ]:
GEOM = ("inc_disc", "pa_disc", "inc_out", "pa_out")
free = lambda keys: [k for k in keys if not np.isscalar(MODEL_PARAMS[k])]
disc_fits, outflow_fits, combined_fits = {}, {}, {}

if RUN_KINEMATIC_FITS and DISC_LINES:
    # 4.1 disc-only fits to the disc-dominated lines
    disc_fits = proj.checkpoint("step2a_disc_fits", btb.fit_many, line_maps, DISC_LINES, "disc",
                                params=MODEL_PARAMS, **MCMC, rerun=RERUN_KINEMATICS,
                                settings=dict(lines=DISC_LINES, params=MODEL_PARAMS, **MCMC))
    n = DISC_LINES[0]
    fig = bplt.plot_kinematic_fit(disc_fits[n], line_maps[n]);  proj.savefig(fig, f"disc_fit_{n}")

In [ ]:
if RUN_KINEMATIC_FITS and OUTFLOW_LINES:
    # 4.2 outflow-only fits to the outflow-dominated lines
    outflow_fits = proj.checkpoint("step2b_outflow_fits", btb.fit_many, line_maps, OUTFLOW_LINES, "outflow",
                                   params=MODEL_PARAMS, **MCMC, rerun=RERUN_KINEMATICS,
                                   settings=dict(lines=OUTFLOW_LINES, params=MODEL_PARAMS, **MCMC))
    n = OUTFLOW_LINES[-1]
    fig = bplt.plot_kinematic_fit(outflow_fits[n], line_maps[n]);  proj.savefig(fig, f"outflow_fit_{n}")

In [ ]:
if RUN_KINEMATIC_FITS:
    # 4.3 combined fits to every decomposable line, started near the single-component results
    start = {**btb.mean_params(disc_fits, free(("v_disc", "R_disc") + GEOM[:2])),
             **btb.mean_params(outflow_fits, free(("v_out", "R_out", "theta_in", "theta_out") + GEOM[2:]))}
    COMBINED_PARAMS = dict(MODEL_PARAMS)
    if FIX_GEOMETRY_IN_COMBINED:
        COMBINED_PARAMS.update({k: start[k] for k in GEOM if k in start})
    init = {k: (0.7 * v, 1.3 * v + 1) for k, v in start.items() if k not in GEOM}
    combined_fits = proj.checkpoint("step2c_combined_fits", btb.fit_many, line_maps, DECOMPOSE_LINES,
                                    "combined", params=COMBINED_PARAMS, init=init, **MCMC, rerun=RERUN_KINEMATICS,
                                    settings=dict(lines=DECOMPOSE_LINES, params=COMBINED_PARAMS, init=init, **MCMC))
    btb.fits_table(combined_fits)
    n = DECOMPOSE_LINES[len(DECOMPOSE_LINES) // 2]
    fig = bplt.plot_kinematic_fit(combined_fits[n], line_maps[n]);  proj.savefig(fig, f"combined_fit_{n}")

Parameters against IP (box: 25-75th percentile, whiskers: 16-84th, line: median; least-squares fits show error bars).
Parameters that are roughly constant are averaged for the decomposition; any that change systematically with IP (in
NGC 5728: $R_{\rm out}$, $\alpha$) get a linear trend. **Check the plot** before trusting a trend.

In [ ]:
if RUN_KINEMATIC_FITS:
    fig = bplt.plot_params_vs_ip(combined_fits, ["R_out", "alpha", "v_disc", "v_out"], style="box")
    proj.savefig(fig, "params_vs_IP")
    FIXED_IN_FITS = {k: v for k, v in COMBINED_PARAMS.items() if np.isscalar(v)}          # fixed values
    FIT_MEANS = btb.mean_params(combined_fits, [k for k in COMBINED_PARAMS if k != "alpha"])  # fitted: averaged
    R_OUT_TREND = (btb.trend_with_ip(combined_fits, "R_out")
                   if (not np.isscalar(COMBINED_PARAMS["R_out"]) and len(combined_fits) > 2) else None)

**Best-fit geometry.** Schematic of the disc and bicone from the combined fits (averaged over the lines; cones drawn out
to 2 $R_{\rm out}$). Left: as seen by the observer - parts of the cones behind the disc are drawn as a wire-frame, parts in
front are shaded, yellow lines mark where the cones cross the disc plane. Right: a 3-D view with the line of sight.
Check that it is physically sensible (e.g. the cone that is redshifted points away from us). `DISC_NEAR_SIDE` flips
which side of the disc is nearer to us.

In [ ]:
if RUN_KINEMATIC_FITS:
    FIT_GEOMETRY = btb.mean_geometry(combined_fits)
    fig = btb.plot_geometry(FIT_GEOMETRY, disc_near_side=DISC_NEAR_SIDE, title=f"{GALAXY}: best-fit geometry")
    proj.savefig(fig, "geometry_best_fit")

### Optional: velocity / dispersion maps from other codes (3DBarolo, MOKA3D, ...)

Any component can use a map produced elsewhere instead of the built-in model. Give the AGN pixel position in that map,
its pixel size, and the systemic velocity to subtract if it holds absolute velocities. Keys: `"v_disc"`, `"v_out"`,
`"sigma_disc"`, `"sigma_out"`. Where an imported map is undefined the built-in model / fixed dispersion is used.

In [ ]:
USE_EXTERNAL_MAPS = False
EXTERNAL = {}

if USE_EXTERNAL_MAPS:
    pc_scale = next(iter(cubes.values())).pc_per_arcsec
    EXTERNAL["v_disc"] = btb.load_external_map("path/to/model_velocity.fits",   # EDIT
                                               agn_xy=(0.0, 0.0),                # EDIT: centre in that map
                                               pixel_scale_arcsec=0.2,           # EDIT (or read from the header)
                                               pc_per_arcsec=pc_scale, subtract=0.0)
    lm = line_maps[DECOMPOSE_LINES[0]]
    fig = bplt.plot_velocity_panels({"observed": lm.velocity_map(),
                                     "imported disc": np.where(lm.good, EXTERNAL["v_disc"].on_grid(lm), np.nan)},
                                    lm.extent_pc(), vlim=400)
    proj.savefig(fig, "external_map_check")

## 5. Step 3 - constrained decomposition - EDIT

`DECOMP_PARAMS` selects where each line's model parameters come from:

* `"fit_mean"` (default) - fitted geometry + averages of the combined fits, with $R_{\rm out}$ from its trend with IP
  (set `USE_R_OUT_TREND = False` to use the average instead);
* `"fit_per_line"` - each line's own combined fit (more freedom; check weak lines carefully);
* `"fixed_common"` - your own values in `FIXED_COMMON`, the same for every line;
* `"fixed_per_line"` - your own values per line in `FIXED_LINE_PARAMS` (radii in pc).

`DECOMP_SIGMAS`: `"measured"` (Step 1 medians) or `"fixed"` (`FIXED_SIGMAS`).

In [ ]:
DECOMP_PARAMS   = "fit_mean"     # "fit_mean" | "fit_per_line" | "fixed_common" | "fixed_per_line"
DECOMP_SIGMAS   = "measured"     # "measured" | "fixed"
USE_R_OUT_TREND = True

# Only used with the "fixed_*" options - EDIT with your own values
FIXED_COMMON = dict(v_sys=0.0, inc_disc=None, pa_disc=None, v_disc=None, R_disc=None,    # km/s, deg, pc
                    inc_out=None, pa_out=None, v_out=None, R_out=None, theta_in=None, theta_out=None)
FIXED_LINE_PARAMS = {}           # {"line": {"v_disc": ..., "R_disc": ..., ...}} - overrides FIXED_COMMON per line
FIXED_SIGMAS = {}                # {"line": (sigma_disc, sigma_outflow)} [km/s]

# Optional: decompose spectra re-extracted with other windows / continuum, e.g.
#   {"NeII": dict(window=(12.72, 12.92), line_region=(12.785, 12.835), cont_order=1)}
DECOMP_RECUT = {}

In [ ]:
dmaps = {n: line_maps[n] for n in DECOMPOSE_LINES}
if DECOMP_PARAMS == "fit_mean":
    rel = {"R_out": (lambda ip, a=R_OUT_TREND[0], b=R_OUT_TREND[1]: a * ip + b)} \
          if (USE_R_OUT_TREND and R_OUT_TREND is not None) else None
    line_parameters = btb.line_params(dmaps, {**FIXED_IN_FITS, **FIT_MEANS}, ip_relations=rel)
elif DECOMP_PARAMS == "fit_per_line":
    line_parameters = btb.line_params(dmaps, {}, fits=combined_fits)
elif DECOMP_PARAMS in ("fixed_common", "fixed_per_line"):
    per_line = FIXED_LINE_PARAMS if DECOMP_PARAMS == "fixed_per_line" else None
    line_parameters = btb.line_params(dmaps, FIXED_COMMON, per_line=per_line)
    missing = {n: [k for k, v in p.items() if v is None] for n, p in line_parameters.items()}
    if any(missing.values()):
        raise ValueError(f"Fill in FIXED_COMMON / FIXED_LINE_PARAMS first - missing: {missing}")
else:
    raise ValueError(DECOMP_PARAMS)

if DECOMP_SIGMAS == "fixed":
    sigmas = {n: {"disc": FIXED_SIGMAS[n][0], "outflow": FIXED_SIGMAS[n][1]} for n in DECOMPOSE_LINES}
else:
    sigmas = sigmas_measured

proj.save_config(decomp_params_source=DECOMP_PARAMS, decomp_sigmas_source=DECOMP_SIGMAS,
                 decomposition_params=line_parameters, decomposition_sigmas=sigmas)

In [ ]:
DECOMP_OPTIONS = dict(names=DECOMPOSE_LINES, sigma_mode="fixed", recut=DECOMP_RECUT or None)

decomps = proj.checkpoint("step3_decompositions", btb.decompose_lines, line_maps, line_parameters, sigmas,
                          cubes=cubes, external=EXTERNAL or None, **DECOMP_OPTIONS, rerun=RERUN_DECOMPOSITION,
                          settings=dict(params=line_parameters, sigmas=sigmas, external=sorted(EXTERNAL),
                                        **DECOMP_OPTIONS))

Disc-fraction map of every line, and the residual maps: **coherent residual structure** means the two-component
model does not describe that line (a third component, wrong velocities/dispersions, or a blend).

In [ ]:
fig = bplt.plot_fraction_maps(decomps, "f_disc");  proj.savefig(fig, "disc_fraction_maps")
fig = bplt.plot_residual_maps(decomps);            proj.savefig(fig, "residual_maps")

## 6. Step 4 - stack all lines into one map

Per-line $f_{\rm disc}$ maps on a common parsec grid, weighted by $w = 1 - 2|\langle f_{\rm disc}\rangle - 0.5|$, keeping
points covered by at least `min_lines` lines. `MAP_LIM_PC` / `SCALEBAR_PC` only affect the figure.

In [ ]:
STACK_OPTIONS = dict(scheme="linear", combine="mean", min_lines=2)
MAP_LIM_PC  = None       # e.g. 1500 to zoom the figure
SCALEBAR_PC = None       # e.g. 500 for a scale bar
SF_MAX, AGN_MIN = 0.2, 0.8      # f_out thresholds for disc- / outflow-dominated (also used for the spectra)

stack = btb.stack_fraction_maps(decomps, **STACK_OPTIONS)
print(f"line-to-line scatter in f_disc: {btb.line_to_line_scatter(stack):.2f}")
FRACTIONS = btb.fraction_summary(stack, sf_max=SF_MAX, agn_min=AGN_MIN)
proj.save_config(fraction_summary=FRACTIONS)

fig = bplt.plot_stacked_map(stack, which="f_out", lim_pc=MAP_LIM_PC, scalebar_pc=SCALEBAR_PC, title=GALAXY)
proj.savefig(fig, "outflow_fraction_map")
proj.save(stack, "step4_stacked_map")
proj.export_stacked_map(stack)

**Geometry used in the decomposition** (averaged over the decomposed lines). With per-line parameters this is their
mean; if imported maps replaced a component (Section 4), its drawn geometry is only indicative.

In [ ]:
USED_GEOMETRY = btb.mean_geometry(line_parameters)
fig = btb.plot_geometry(USED_GEOMETRY, disc_near_side=DISC_NEAR_SIDE, title=f"{GALAXY}: geometry used in the decomposition")
proj.savefig(fig, "geometry_used")

**Robustness**: the map should not depend strongly on the weighting or on any single line.

In [ ]:
alternatives = {"linear weights": stack,
                "equal weights": btb.stack_fraction_maps(decomps, **{**STACK_OPTIONS, "scheme": "equal"}, verbose=False),
                "median": btb.stack_fraction_maps(decomps, **{**STACK_OPTIONS, "combine": "median"}, verbose=False)}
for n in DECOMPOSE_LINES:                                       # leave each line out in turn
    alternatives[f"without {n}"] = btb.stack_fraction_maps(decomps, names=[m for m in DECOMPOSE_LINES if m != n],
                                                           **STACK_OPTIONS, verbose=False)
btb.compare_stacks(alternatives, "linear weights")

## 7. Step 5 - AGN and star-forming spectra

Spaxels with $f_{\rm out} > $ `AGN_MIN` give the AGN-dominated spectrum, $f_{\rm out} < $ `SF_MAX` the star-formation-
dominated one (`method="weighted"` weights every spaxel by its fractions instead). Check that both regions contain
spaxels in the extraction plot.

**Errors** (third column of the exported `.txt` files, 1$\sigma$ in Jy, shaded in the plot): `err_method="cube"` adds the
pipeline ERR values of the selected spaxels in quadrature - a lower limit, because neighbouring spaxels of a resampled
cube share noise; `"empirical"` measures the channel-to-channel scatter of each extracted spectrum (includes correlated
noise, inflated next to strong lines); `"max"` takes the larger. The band-to-band scale factors' uncertainty is not included.

In [ ]:
SPECTRA_OPTIONS = dict(method="threshold", agn_min=AGN_MIN, sf_max=SF_MAX, min_spaxels=3,
                       err_method="cube")   # errors: "cube" (ERR, quadrature), "empirical", "max" or None

spectra = btb.extract_all(cubes, stack, **SPECTRA_OPTIONS)
CHECK_BAND = lines[DECOMPOSE_LINES[0]].cube                    # EDIT to look at another band
fig = bplt.plot_extraction_regions(spectra[CHECK_BAND], cubes[CHECK_BAND]);  proj.savefig(fig, "extraction_regions")

In [ ]:
w_agn, f_agn, e_agn, _ = btb.stitch_spectra(spectra, "agn", with_errors=True)
w_sf,  f_sf,  e_sf,  _ = btb.stitch_spectra(spectra, "sf",  with_errors=True)
print(f"median S/N per channel: AGN {np.nanmedian(f_agn / e_agn):.0f}, SF {np.nanmedian(f_sf / e_sf):.0f}")

ERR_DISPLAY_SCALE = 1       # exaggerate the shaded errors for display only (e.g. 10 if they are invisible)
fig = bplt.plot_component_spectra(w_agn, f_agn, w_sf, f_sf, smooth=2, err_agn=e_agn, err_sf=e_sf,
                                  err_scale=ERR_DISPLAY_SCALE)
proj.savefig(fig, "component_spectra")
proj.save({"agn": (w_agn, f_agn, e_agn), "sf": (w_sf, f_sf, e_sf), "per_band": spectra}, "step5_spectra")
proj.export_spectra({"AGN": (w_agn, f_agn, e_agn), "SF": (w_sf, f_sf, e_sf)})   # wavelength, flux, error

In [ ]:
proj.summary()

## 8. Corrected spectra: extinction, then the AGN PSF (AGN point source, AGN outflow and star formation)

The Step 5 spectra (Section 7) are sums of observed spaxels. Two effects remain in them: **extinction**, which differs
from spaxel to spaxel, and the **unresolved AGN**, whose light the point-spread function (PSF) spreads over the
AGN-dominated region and - through its wings - over the SF-dominated region. This section removes both, spaxel by
spaxel and band by band, in that order, and only then stitches the bands:

| Step | What | Output |
|---|---|---|
| 8.1 | H$_2$ S(1)-S(7) line maps, PSF-matched on the MRS channel-3 grid of S(1) | |
| 8.2 | $\tau_{9.8}$ in every spaxel from the dip of S(3) in the rotation diagram (Donnan et al. 2024, Fig. 7) | example excitation diagrams |
| 8.3 | **$\tau_{9.8}$ map** on the grid and field of the AGN outflow fraction map | `tau98_map.pdf`, `.npz`, `.fits` |
| 8.4 | Extinction correction of every AGN- and SF-dominated spaxel, per band | extinction-corrected spectra per band |
| 8.5 | Nuclear spectrum in a 0.3" aperture on the AGN (continuum peak) | AGN-raw |
| 8.6 | AGN point source and PSF model in every band (STPSF), with two checks: PSF contamination maps, curves of growth | point-source spectrum S, PSF per spaxel |
| 8.7 | PSF contamination of every extinction-corrected spaxel removed; spaxels summed per band | corrected spectra per band |
| 8.8 | Bands stitched from the short-wavelength end; final spectra and their comparison with the raw ones | `spectrum_*.txt` |

**Why extinction first.** $\tau_{9.8}$ is measured from the H$_2$ rotational lines, which the AGN does not emit, so it is
derived from the observed cubes with no PSF model involved - the PSF-modelling uncertainties cannot enter it. Each
spaxel of the AGN- and SF-dominated regions (Step 5) is then corrected for extinction, $D_i'(\lambda) = D_i(\lambda)\,e^{+\tau_{9.8,i}\tau(\lambda)}$,
and the AGN light it contains is removed. The AGN light in spaxel $i$ is $P_i(\lambda)\,S(\lambda)$ - $P_i$ the fraction of the AGN point
source that the PSF model puts in the spaxel, $S$ the point-source spectrum of that band - and the extinction correction
has scaled it by the same factor, so the PSF contamination of the corrected spaxel is $C_i = P_i\,S\,e^{+\tau_{9.8,i}\tau(\lambda)}$:

$$F_R(\lambda) = \sum_{i \in R} \big[D_i'(\lambda) - C_i(\lambda)\big].$$

Every correction is done per band before stitching, which avoids the jumps at the band edges that appear when
band-dependent corrections are applied to an already stitched spectrum.

**Extinction.** A foreground screen $F_{\rm obs} = F_{\rm int}\,e^{-\tau_{9.8}\tau(\lambda)}$ with the extinction law of
Donnan et al. (2023a) (`EXT_LAW = "donnan2023"`: $0.1(9.8/\lambda)^{1.7} + 0.9\,\tau_{\rm Sil}$, the empirical silicate profile of
IRAS 08572+3915 with crystalline features; the law used by Donnan et al. 2024). The H$_2$ S(3) line (9.665 $\mu$m) sits in
the 9.8 $\mu$m silicate absorption, so $\tau_{9.8}$ is the value that puts S(3) back in line with the other transitions:
`TAU_METHOD = "powerlaw"` (S(3) on the power law $N_{\rm u}/g_{\rm u} \propto E_{\rm u}^{\,b}$ through the other corrected transitions)
or `"smoothness"` (minimum Laplacian penalty of the corrected diagram, Donnan et al. 2024 eq. 18).

**Final spectra** (saved as `spectrum_<label>.txt`: wavelength, flux, random error, PSF-model systematic):
**AGN-raw** (0.3" aperture), **AGN-psf-corrected** (the aperture spectrum divided by the PSF encircled energy of the
aperture in each band: the total flux of the AGN point source; not extinction-corrected), **AGN-outflow-corrected** and
**SF-corrected** (the region spectra above). The uncorrected AGN- and SF-dominated spectra are those of Section 7
(`component_spectra_AGN.txt`, `component_spectra_SF.txt`).

**Caveats.** (i) The extinction correction assumes everything in a spaxel (continuum, PAHs, ionised lines) is behind the
same screen as the warm H$_2$; Donnan et al. find that different components can see different extinction. (ii) The
ortho-to-para ratio is assumed to be 3: a zig-zag in the example diagrams (odd J high, even J low) biases $\tau_{9.8}$.
(iii) S(7) is blended with [Mg VII] 5.503 $\mu$m in AGN; it is deblended with Gaussians tied to the kinematics of the
nearest unblended H$_2$ line (leave it out with `H2_FIT_LINES`). (iv) Where extended emission around the nucleus is as
compact as the PSF (long wavelengths), no method can separate it from the point source (check 2 in 8.6). (v) Each band
sees the part of a region inside its own field; with `STITCH_REF = "blue"` the spectrum is normalised to the bluest band
kept, so bands covering less than `MIN_AREA_FRAC` of the largest region area are dropped and the scale factors printed
(factors far from 1 mean the bands see different parts of the region). The Section 7 spectra are stitched from the band
with the largest area instead, so their normalisation can differ from the comparison spectra shown here.

In [ ]:
# ---- 8.1-8.3 extinction from H2 (tau_9.8 from the observed cubes; no PSF model involved)
H2_LINES         = ["S1", "S2", "S3", "S4", "S5", "S6", "S7"]   # H2 0-0 S(J) transitions (all MRS)
H2_TARGET        = "S1"          # common grid: the band holding S(1) (MRS channel 3) - or a cube name, e.g. "ch3-long"
H2_MATCH_PSF     = True          # convolve every line to the PSF FWHM at S(1) before resampling
H2_HALFWIDTH     = 700           # half-width of the integrated line region [km/s] (continuum: 1200 km/s on each side)
H2_SNR_MIN       = 3             # transitions below this S/N are not used in a spaxel
H2_FIT_LINES     = None          # transitions used for tau_9.8 (None = all H2_LINES), e.g. without "S7"
EXT_LAW          = "donnan2023"  # tau(lambda)/tau_9.8: "donnan2023" (Donnan+23a), "kvt" (PAHFIT) or a (wave, tau) table
TAU_METHOD       = "powerlaw"    # "powerlaw" (S(3) on the power law through the other lines) or "smoothness" (Donnan+24)
TAU_SYS          = 0.05          # fractional flux uncertainty added in quadrature (calibration, PSF matching)
TAU_N_MC         = 200           # Monte-Carlo draws for the tau_9.8 uncertainty
N_EXAMPLES       = 5             # example spaxels (spanning the tau_9.8 range) ...
EXAMPLE_SPAXELS  = None          # ... or your own: [(iy, ix), ...] on the H2_TARGET grid
# ---- 8.4 extinction correction per spaxel
TAU_FILL         = "nearest"     # spaxels without tau_9.8: "nearest", "median", "zero" (no correction) or a number
TAU_CLIP_NEG     = True          # negative tau_9.8 (S(3) above the trend: noise) -> 0
TAU_SMOOTH       = 0.0           # Gaussian smoothing of the tau_9.8 map before applying it [arcsec FWHM]
TAU_MAX_ERR      = None          # treat spaxels with sigma(tau_9.8) above this as unmeasured (then filled)
# ---- 8.5 nuclear aperture
NUCLEAR_RADIUS   = 0.3           # aperture radius [arcsec], centred on the AGN (continuum peak) of each band
APERTURE_METHOD  = "exact"       # "exact" (fractional pixel weights) or "center" (whole pixels)
APERTURE_ERR_METHOD = SPECTRA_OPTIONS.get("err_method", "cube")   # same as Step 5
# ---- 8.6-8.7 AGN point source, PSF model and PSF correction
PSF_MODEL        = "webbpsf"     # "webbpsf" (STPSF / WebbPSF instrument PSFs) or "gaussian"
PSF_POINT_SOURCE = "fit"         # point-source spectrum S: "fit" (PSF + smooth extended emission fitted around the
                                 # nucleus; recommended) or "aperture" (aperture flux / EE: all light in it is AGN)
PSF_FIT_RADIUS   = None          # "fit": radius of the fitted region [arcsec] (default max(0.6", 2 x FWHM))
PSF_SUBTRACT     = "continuum"   # removed from the regions: "continuum" (smooth AGN continuum; lines, PAH, H2 kept)
                                 # or "full" (per-channel point-source spectrum incl. its lines; noisier)
PSF_ANNULUS      = None          # "aperture" only: e.g. (0.6, 0.9) arcsec - estimate the extended emission there
PSF_SYMMETRIC    = True          # azimuthally average the PSF (orientation of the sky-aligned cubes does not matter)
PSF_N_WAVE       = 6             # PSF sample wavelengths per band (interpolated in between)
PSF_SYS          = 0.10          # PSF-model uncertainty, as a fraction of each PSF correction
FWHM_SCALE       = 1.0           # "gaussian" only: scale the FWHM (e.g. 1.1 to mimic cube-building broadening)
# ---- 8.8 stitching
STITCH_REF       = "blue"        # "blue": bluest usable band unscaled, then band by band to longer wavelengths;
                                 # "largest": the band with the largest region area; or a band name
MIN_AREA_FRAC    = 0.3           # region spectra: drop bands covering less than this fraction of the largest area

### 8.1 H$_2$ line maps

Continuum-subtracted flux maps of S(1)-S(7), each in the band that covers it (top: native grids; grey: S/N <
`H2_SNR_MIN`). Below: every line convolved to the PSF at S(1) (the coarsest) and resampled (flux conserving) onto the
channel-3 grid of S(1), where the rotation diagram of each spaxel is built. The channel-1 bands of S(5)-S(7) cover a
smaller field: outside it a spaxel has S(1)-S(4) only, which is still enough (S(3) needs a transition on each side).

In [ ]:
fig = bplt.plot_extinction_law(EXT_LAW, H2_LINES)
proj.savefig(fig, "extinction_law")
h2_native = btb.h2_line_maps(cubes, H2_LINES, target=None, match_psf=False, halfwidth_kms=H2_HALFWIDTH)
fig = bplt.plot_h2_maps(h2_native, snr_min=H2_SNR_MIN)
proj.savefig(fig, "h2_maps_native")
h2_maps = btb.h2_line_maps(cubes, H2_LINES, target=H2_TARGET, match_psf=H2_MATCH_PSF, halfwidth_kms=H2_HALFWIDTH)
fig = bplt.plot_h2_maps(h2_maps, snr_min=H2_SNR_MIN)
proj.savefig(fig, "h2_maps_common_grid")

### 8.2 $\tau_{9.8}$ in every spaxel

Top: observed (black) and extinction-corrected (red) rotation diagrams of example spaxels spanning the $\tau_{9.8}$ range;
red dashed: the power law through the corrected transitions other than S(3). Bottom: the quantity minimised against
$\tau_{9.8}$ (as Donnan et al. 2024 Fig. 7), with the best value (solid) and the 16-84 per cent range (dashed). Then the
$\tau_{9.8}$ map on the H$_2$ grid, its uncertainty and the S/N of S(3), with the example spaxels numbered and the
$f_{\rm out}$ = `SF_MAX` (orange) and `AGN_MIN` (blue) contours. As a check, the other method is run too.

In [ ]:
tau_map = btb.fit_tau(h2_maps, method=TAU_METHOD, law=EXT_LAW, snr_min=H2_SNR_MIN, sys_frac=TAU_SYS,
                      n_mc=TAU_N_MC, fit_lines=H2_FIT_LINES)
examples = EXAMPLE_SPAXELS or btb.example_spaxels(tau_map, N_EXAMPLES)
fig = bplt.plot_excitation_diagrams(tau_map, examples)
proj.savefig(fig, "h2_excitation_diagrams")
for j, (iy, ix) in enumerate(examples):
    print(f"#{j + 1} spaxel (iy, ix) = ({iy}, {ix}): tau_9.8 = {tau_map.tau[iy, ix]:.2f} "
          f"(+{tau_map.tau_hi[iy, ix] - tau_map.tau[iy, ix]:.2f} / -{tau_map.tau[iy, ix] - tau_map.tau_lo[iy, ix]:.2f}), "
          f"{tau_map.n_lines[iy, ix]} transitions, S(3) S/N = {tau_map.snr_s3[iy, ix]:.0f}")
fig = bplt.plot_tau_map(tau_map, examples, stack=stack, levels=(SF_MAX, AGN_MIN))
proj.savefig(fig, "tau98_map_h2_grid")

other = "smoothness" if TAU_METHOD == "powerlaw" else "powerlaw"
tau_alt = btb.fit_tau(h2_maps, method=other, law=EXT_LAW, snr_min=H2_SNR_MIN, sys_frac=TAU_SYS, n_mc=20,
                      fit_lines=H2_FIT_LINES, verbose=False)
both = tau_map.ok & tau_alt.ok
if both.any():
    d = tau_alt.tau[both] - tau_map.tau[both]
    print(f"check - '{other}' minus '{TAU_METHOD}': median {np.median(d):+.2f}, scatter "
          f"{1.4826 * np.median(np.abs(d - np.median(d))):.2f} (median uncertainty of '{TAU_METHOD}': "
          f"{np.nanmedian(tau_map.tau_err[both]):.2f})")

### 8.3 The $\tau_{9.8}$ map

The $\tau_{9.8}$ map on the same grid and field as the AGN outflow fraction map (Section 6), area-averaged from the H$_2$
grid; grey: field of the outflow map without a $\tau_{9.8}$ measurement (no filling here). Contours: $f_{\rm out}$ = `SF_MAX`
(orange) and `AGN_MIN` (blue). Saved as `figures/tau98_map.pdf` (and `.png`) and exported as `tau98_map.npz` / `.fits`
(same pc axes as `stacked_fraction_map`).

In [ ]:
tau_stack = btb.tau_on_stack(tau_map, stack, clip_negative=TAU_CLIP_NEG)
tau_stack_err = btb.tau_on_stack(tau_map, stack, which="tau_err")
fig = bplt.plot_tau98_on_stack(stack, tau_stack, lim_pc=MAP_LIM_PC, levels=(SF_MAX, AGN_MIN), scalebar_pc=SCALEBAR_PC,
                               title=f"{GALAXY}: $\\tau_{{9.8}}$ from H$_2$")
proj.savefig(fig, "tau98_map", formats=("pdf", "png"))
proj.export_tau_map(stack, tau_stack, tau_stack_err)
for lab, sel in (("AGN-dominated", stack.f_out > AGN_MIN), ("SF-dominated", stack.f_out < SF_MAX)):
    v = tau_stack[sel & np.isfinite(tau_stack)]
    if v.size:
        print(f"{lab} region: median tau_9.8 = {np.median(v):.2f} (16-84%: {np.percentile(v, 16):.2f}-"
              f"{np.percentile(v, 84):.2f}; measured in {100 * v.size / max(sel.sum(), 1):.0f}% of its pixels)")

### 8.4 Extinction correction of every spaxel

The $\tau_{9.8}$ map resampled onto the grid of every band (area-averaged; spaxels without a measurement filled with
`TAU_FILL`, negative values set to 0 with `TAU_CLIP_NEG`), and every AGN- and SF-dominated spaxel corrected with
$e^{+\tau_{9.8,i}\tau(\lambda)}$. Printed: the median $\tau_{9.8}$ of each region per band. Below: $\tau_{9.8}$ on three bands with the AGN (blue) and
SF (orange) spaxels outlined, and the extinction-corrected region spectra of every band (not yet PSF-corrected or stitched).

In [ ]:
tau_on = {b: btb.tau_on_cube(tau_map, cubes[b], fill=TAU_FILL, clip_negative=TAU_CLIP_NEG,
                             smooth_fwhm_arcsec=TAU_SMOOTH, max_err=TAU_MAX_ERR) for b in spectra if b in cubes}
corr_ext = btb.corrected_region_spectra(cubes, spectra, {}, tau_maps=tau_on, law=EXT_LAW, verbose=False)  # extinction only

bands = [b for b in sorted(spectra, key=lambda b: cubes[b].wave_range[0]) if b in tau_on]
show = [bands[0], tau_map.grid.name if tau_map.grid.name in bands else bands[len(bands) // 2], bands[-1]]
vmax = np.nanpercentile(np.concatenate([tau_on[b][spectra[b].footprint].ravel() for b in show]), 99)
fig, axes = plt.subplots(1, 5, figsize=(26, 4.8), constrained_layout=True,
                         gridspec_kw=dict(width_ratios=[1, 1, 1, 1.6, 1.6]))
for ax, b in zip(axes[:3], show):
    bplt.style_axes(ax)
    sp_ = spectra[b]
    im = ax.imshow(np.where(sp_.footprint, tau_on[b], np.nan), cmap="magma", vmin=0, vmax=vmax)
    for m, col in ((sp_.agn_mask, "deepskyblue"), (sp_.sf_mask, "darkorange")):
        if m is not None and m.any():
            ax.contour(m.astype(float), levels=[0.5], colors=col, linewidths=1.5)
    ax.plot(*cubes[b].agn_xy, "x", color="w", ms=10, mew=2)
    ax.set_title(f"{b} ({cubes[b].pixel_scale:.2f}\"/pix)", fontsize=12); ax.set_xlabel("x [pixel]")
axes[0].set_ylabel("y [pixel]")
bplt.style_colorbar(plt.colorbar(im, ax=axes[2]), r"$\tau_{9.8}$ applied")
cmap_b = plt.get_cmap("turbo")
for ax, (title, bands_) in zip(axes[3:], [("AGN-dominated, extinction-corrected, per band", corr_ext["AGN-ext"]),
                                          ("SF-dominated, extinction-corrected, per band", corr_ext["SF-ext"])]):
    bplt.style_axes(ax)
    names_ = sorted(bands_, key=lambda b: cubes[b].wave_range[0])
    allf = np.concatenate([np.asarray(bands_[b].flux, float) for b in names_]) if names_ else np.ones(1)
    for j, b in enumerate(names_):
        ax.plot(bands_[b].wave, bands_[b].flux, lw=0.8, color=cmap_b(j / max(len(names_) - 1, 1)), label=b)
    ax.set_yscale("log" if np.nanmean(allf <= 0) < 0.05 else "linear"); ax.set_title(title, fontsize=12)
    ax.set_xlabel(r"Rest-frame wavelength [$\mu$m]"); ax.legend(fontsize=7, ncol=2)
axes[3].set_ylabel("Flux density [Jy]")
proj.savefig(fig, "extinction_corrected_per_band")
for b in bands:
    sp_ = spectra[b]
    med = [f"{R} {np.median(tau_on[b][m]):.2f}" for R, m in (("AGN-dominated", sp_.agn_mask), ("SF-dominated", sp_.sf_mask))
           if m is not None and m.any()]
    print(f"{b:>14s}: median tau_9.8 applied: " + ", ".join(med))

### 8.5 Nuclear spectrum

A 0.3" radius aperture centred on the AGN position of every band (exact fractional pixel weights, the Step 5 error
method, detector-gap and field-of-view handling; a band is skipped if less than 90% of the aperture lies inside its
field). It is not PSF-corrected: a fixed aperture loses a growing fraction of a point source's flux towards long
wavelengths (MRS FWHM ~0.3" at 5 $\mu$m, ~1" at 28 $\mu$m) - that is done in 8.6.

In [ ]:
APERTURES = {"nuclear": dict(radius_arcsec=NUCLEAR_RADIUS, offset_arcsec=(0.0, 0.0))}
ap_spectra = btb.extract_apertures(cubes, APERTURES, method=APERTURE_METHOD, err_method=APERTURE_ERR_METHOD)
PC_PER_ARCSEC = next(iter(cubes.values())).pc_per_arcsec
fig = bplt.plot_apertures_on_map(stack, APERTURES, PC_PER_ARCSEC, which="f_out", lim_pc=MAP_LIM_PC,
                                 title=f"{GALAXY}: nuclear aperture")
proj.savefig(fig, "nuclear_aperture_on_outflow_map")

### 8.6 AGN point source and PSF model

The AGN is unresolved. For every band the PSF model gives, as a function of wavelength, the fraction of the point
source in every spaxel - and so in the nuclear aperture (EE) and in the AGN- and SF-dominated regions (plotted below).
The point-source spectrum $S$ of each band comes from `PSF_POINT_SOURCE`: `"fit"` fits the spaxels around the nucleus with
point source x PSF + a smooth extended component at every wavelength (separates the sharp PSF core from extended
emission); `"aperture"` takes aperture flux / EE. With `PSF_SUBTRACT = "continuum"` only a smooth continuum of $S$ (the
unresolved hot-dust continuum) is removed from the regions in 8.7, so their line, PAH and H$_2$ emission is untouched and
almost no noise is added.

**PSF model.** `PSF_MODEL = "webbpsf"` uses STPSF (formerly WebbPSF) monochromatic MIRI/MRS and NIRSpec IFU PSFs - installed
with the environment (`conda env update -f environment.yml --prune` for an existing one); its data files (~90 MB) download
to `~/data/stpsf-data` the first time (or set `STPSF_PATH`; https://stpsf.readthedocs.io). Without STPSF the notebook falls
back to `"gaussian"`: FWHM = 0.033 $\lambda$ + 0.106" (MRS; Argyriou et al. 2023). PSF models do not include the broadening by
cube building, so real cube PSFs are somewhat wider.

In [ ]:
kw = dict(n_wave=PSF_N_WAVE, symmetric=PSF_SYMMETRIC, fwhm_scale=FWHM_SCALE, psf_sys=PSF_SYS, annulus_arcsec=PSF_ANNULUS,
          point_source=PSF_POINT_SOURCE, fit_radius_arcsec=PSF_FIT_RADIUS, subtract=PSF_SUBTRACT)
try:
    psf_res, psf_frac = btb.psf_correct(cubes, spectra, ap_spectra["nuclear"], model=PSF_MODEL, **kw)
except ImportError as ex:
    print(f"{ex}\n-> falling back to the Gaussian PSF model")
    PSF_MODEL = "gaussian"
    psf_res, psf_frac = btb.psf_correct(cubes, spectra, ap_spectra["nuclear"], model=PSF_MODEL, **kw)
fig = bplt.plot_psf_fractions(psf_frac)
fig.axes[0].set_title(f"PSF model: {PSF_MODEL}")
proj.savefig(fig, "psf_fractions")

**Check 1 - PSF contamination maps.** On the native spaxel grid of the band covering each wavelength: top, the share of
the AGN point-source flux falling in each spaxel (the PSF model itself); bottom, the share of each spaxel's observed flux
that is AGN PSF light. Blue/red contours: AGN- and SF-dominated regions; white circle: the nuclear aperture. With the
instrument PSF the diffraction wings reach several arcsec, so the SF regions receive a little AGN light - increasingly
towards long wavelengths; a Gaussian PSF has almost no wings.

In [ ]:
CONTAM_WAVES = [3.0, 5.0, 12.0, 21.0]     # rest-frame um (continuum wavelengths)
pkw = dict(model=PSF_MODEL, symmetric=PSF_SYMMETRIC, fwhm_scale=FWHM_SCALE)
contam_maps = [btb.psf_contamination(cubes, psf_res, lam, **pkw) for lam in CONTAM_WAVES]
for m in contam_maps:
    if m is not None:
        sp = spectra.get(m["band"])
        share = lambda mask: np.nansum(np.where(mask, m["psf_pct"], 0)) if mask is not None else np.nan
        print(f"{m['lam']:5.1f} um ({m['band']}): AGN point source {1e3 * m['agn_flux']:.1f} mJy; of it "
              f"{share(sp.agn_mask):.1f}% falls in the AGN-dominated region and {share(sp.sf_mask):.3f}% in the "
              f"SF-dominated region; median contamination of SF spaxels "
              f"{np.nanmedian(m['contam_pct'][sp.sf_mask]) if sp.sf_mask is not None and sp.sf_mask.any() else np.nan:.3f}%")
fig = bplt.plot_psf_contamination(contam_maps, spectra, ap_spectra["nuclear"])
proj.savefig(fig, "psf_contamination_maps")

**Check 2 - is the nucleus a point source?** Curves of growth of the observed continuum around the AGN (solid) against
the model PSF (dashed), normalised at 0.3". If the data grow faster than the PSF, there is extended emission around the
nucleus; with `PSF_POINT_SOURCE = "aperture"` it would be counted as AGN light (the AGN spectrum over-estimated and the
regions over-subtracted) - use `"fit"` (default) or `PSF_ANNULUS`. Where the extended emission is as compact as the PSF
(long wavelengths, FWHM ~1"), the AGN-psf-corrected spectrum is an upper limit and AGN-outflow-corrected a lower limit.

In [ ]:
curves = [btb.growth_curve(cubes, lam, **pkw) for lam in CONTAM_WAVES]
fig = bplt.plot_growth_curves(curves)
fig.axes[0].set_title(f"Curves of growth around the AGN ({PSF_MODEL} PSF)")
proj.savefig(fig, "psf_growth_curves")
for c in curves:
    if c is not None:
        print(f"{c['lam']:5.1f} um ({c['band']}): flux within {c['radii'][-1]:g}\" = {c['data'][-1]:.2f}x the 0.3\" flux "
              f"(point source: {c['psf'][-1]:.2f}x)")

### 8.7 PSF correction of the extinction-corrected spaxels

For every band and every extinction-corrected spaxel of 8.4, the PSF contamination $C_i = P_i\,S\,e^{+\tau_{9.8,i}\tau(\lambda)}$ (the AGN
light the PSF model puts in the spaxel, scaled by the same extinction correction as the spaxel) is removed, and the
spaxels are summed. Printed: the fraction of the AGN point source that falls in each region. The AGN-psf-corrected
spectrum of each band is the nuclear aperture divided by its PSF encircled energy. Below, the corrected spectra of every
band in their own colour, *before* stitching, to judge the band-to-band offsets.

In [ ]:
# extinction correction of every spaxel (tau_on, 8.4), then the PSF contamination of the corrected spaxels removed
corr = btb.corrected_region_spectra(cubes, spectra, psf_res, tau_maps=tau_on, law=EXT_LAW, model=PSF_MODEL,
                                    n_wave=PSF_N_WAVE, symmetric=PSF_SYMMETRIC, fwhm_scale=FWHM_SCALE,
                                    subtract=PSF_SUBTRACT, psf_sys=PSF_SYS)

# AGN: nuclear aperture / PSF encircled energy of the aperture, per band
agn_bands, agn_raw_bands = {}, {}
for b, ap_ in ap_spectra["nuclear"].items():
    if b not in psf_frac or not np.isfinite(ap_.flux).any():
        continue
    ee = np.interp(ap_.wave, psf_frac[b]["wave"], psf_frac[b]["ee"])
    s_ = btb.ApertureSpectrum("AGN-psf-corrected", b, ap_.wave, ap_.flux / ee,
                              None if ap_.flux_err is None else ap_.flux_err / ee, ap_.weights, ap_.center_xy,
                              ap_.radius_arcsec, ap_.pixel_scale, ap_.covered_frac, "psf")
    s_.flux_sys = PSF_SYS * np.abs(ap_.flux / ee - ap_.flux)
    agn_bands[b], agn_raw_bands[b] = s_, ap_

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8), constrained_layout=True)
cmap_b = plt.get_cmap("turbo")
for ax, (title, bands_) in zip(axes, [("AGN-psf-corrected, per band", agn_bands),
                                      ("AGN-outflow-corrected, per band", corr["AGN-corr"]),
                                      ("SF-corrected, per band", corr["SF-corr"])]):
    bplt.style_axes(ax)
    names_ = sorted(bands_, key=lambda b: cubes[b].wave_range[0])
    allf = np.concatenate([np.asarray(bands_[b].flux, float) for b in names_]) if names_ else np.ones(1)
    for j, b in enumerate(names_):
        ax.plot(bands_[b].wave, bands_[b].flux, lw=0.8, color=cmap_b(j / max(len(names_) - 1, 1)), label=b)
    ax.set_yscale("log" if np.nanmean(allf <= 0) < 0.05 else "linear"); ax.set_title(title, fontsize=13)
    ax.set_xlabel(r"Rest-frame wavelength [$\mu$m]")
    ax.legend(fontsize=8, ncol=2)
axes[0].set_ylabel("Flux density [Jy]")
proj.savefig(fig, "corrected_spectra_per_band")

### 8.8 Stitching and the final spectra

Each spectrum is stitched from the short-wavelength end (`STITCH_REF`): the bluest usable band is kept as it is and every
following band is scaled to match the spectrum built so far in their overlap (ratio of the median fluxes), so all band
edges join. The factors are printed per spectrum. The raw region spectra shown for comparison are the same spaxels
without corrections, stitched the same way. Right: corrected / raw, with the PSF-only and extinction-only corrections.

In [ ]:
FINAL = {}
for label, bands_, minfrac in [("AGN-raw", agn_raw_bands, 0.0), ("AGN-psf-corrected", agn_bands, 0.0),
                               ("AGN-dominated", corr["AGN-raw"], MIN_AREA_FRAC),
                               ("AGN-outflow_psf-only", corr["AGN-psf"], MIN_AREA_FRAC),
                               ("AGN-dominated_ext-only", corr["AGN-ext"], MIN_AREA_FRAC),
                               ("AGN-outflow-corrected", corr["AGN-corr"], MIN_AREA_FRAC),
                               ("SF-dominated", corr["SF-raw"], MIN_AREA_FRAC),
                               ("SF_psf-only", corr["SF-psf"], MIN_AREA_FRAC),
                               ("SF-dominated_ext-only", corr["SF-ext"], MIN_AREA_FRAC),
                               ("SF-corrected", corr["SF-corr"], MIN_AREA_FRAC)]:
    if not bands_:
        print(f"{label}: no bands"); continue
    print(label)
    try:
        FINAL[label] = btb.stitch_bands(bands_, "flux", reference=STITCH_REF, min_area_frac=minfrac)
    except (ValueError, KeyError) as ex_:
        print(f"  not stitched: {ex_}")

COLF = {"AGN-raw": "0.55", "AGN-psf-corrected": "k", "AGN-dominated": "lightsteelblue",
        "AGN-outflow-corrected": "royalblue", "SF-dominated": "lightsalmon", "SF-corrected": "crimson"}
ROWS = [("AGN-raw", "AGN-psf-corrected", None, None, f"Nuclear {NUCLEAR_RADIUS:g}\" aperture: raw vs PSF-corrected AGN"),
        ("AGN-dominated", "AGN-outflow-corrected", "AGN-outflow_psf-only", "AGN-dominated_ext-only",
         f"AGN-dominated region (f_out > {AGN_MIN:g}): raw vs PSF + extinction corrected"),
        ("SF-dominated", "SF-corrected", "SF_psf-only", "SF-dominated_ext-only",
         f"SF-dominated region (f_out < {SF_MAX:g}): raw vs PSF + extinction corrected")]
fig, axes = plt.subplots(3, 2, figsize=(15, 12.5), constrained_layout=True, gridspec_kw=dict(width_ratios=[2.3, 1]))
for row, (a, b, p_only, e_only, title) in enumerate(ROWS):
    if a not in FINAL or b not in FINAL:
        continue
    ys = "log" if all(np.nanmean(FINAL[k_][1] <= 0) < 0.05 for k_ in (a, b)) else "linear"   # linear if partly negative
    bplt.plot_spectra({a: FINAL[a][:3], b: FINAL[b][:3]}, colors=COLF, ax=axes[row, 0], yscale=ys)
    axes[row, 0].set_title(title, fontsize=13)
    ax = axes[row, 1]; bplt.style_axes(ax)
    wa, fa = FINAL[a][0], FINAL[a][1]
    for k_, lab_, col_, ls_ in ((b, "corrected", COLF[b], "-"), (p_only, "PSF only", "0.4", "--"),
                                (e_only, "extinction only", "darkorange", ":")):
        if k_ is not None and k_ in FINAL:
            ax.plot(FINAL[k_][0], FINAL[k_][1] / np.interp(FINAL[k_][0], wa, fa), color=col_, ls=ls_, lw=1.1, label=lab_)
    ax.axhline(1, color="0.5", ls=":", lw=0.8)
    ax.set_ylabel(f"/ {a}"); ax.set_xlabel(r"Rest-frame wavelength [$\mu$m]")
    ax.legend(fontsize=10)
fig.suptitle(f"Corrected spectra: extinction ({EXT_LAW if isinstance(EXT_LAW, str) else 'custom law'}, {TAU_METHOD}), "
             f"then PSF ({PSF_MODEL}, subtracted: {PSF_SUBTRACT}) per spaxel, stitched from the {STITCH_REF} end",
             fontsize=15)
proj.savefig(fig, "corrected_spectra")

for k in ("AGN-raw", "AGN-psf-corrected", "AGN-outflow-corrected", "SF-corrected"):
    if k not in FINAL:
        continue
    w, f, e, s, sc, ref = FINAL[k]
    np.savetxt(proj.file(f"spectrum_{k}", ".txt"), np.column_stack([w, f, np.nan_to_num(e, nan=0.0), s]),
               header=f"rest_wavelength_um  flux_Jy  err_random_Jy  err_psf_systematic_Jy   (stitched from {ref}; "
                      f"extinction law {EXT_LAW if isinstance(EXT_LAW, str) else 'custom'}, then PSF {PSF_MODEL}/{PSF_SUBTRACT})")
    print(f"exported -> {proj.file(f'spectrum_{k}', '.txt')}")
proj.save({"spectra": FINAL, "per_band": corr, "agn_per_band": agn_bands, "tau_map": tau_map, "tau_on_stack": tau_stack,
           "h2_maps": h2_maps, "psf_fractions": psf_frac,
           "settings": dict(order="extinction, then PSF", nuclear_radius=NUCLEAR_RADIUS, psf_model=PSF_MODEL, psf_point_source=PSF_POINT_SOURCE,
                            psf_subtract=PSF_SUBTRACT, psf_sys=PSF_SYS, law=EXT_LAW, method=TAU_METHOD,
                            h2_lines=H2_LINES, h2_fit_lines=H2_FIT_LINES, tau_fill=TAU_FILL,
                            clip_negative=TAU_CLIP_NEG, smooth=TAU_SMOOTH, stitch_ref=STITCH_REF,
                            min_area_frac=MIN_AREA_FRAC)}, "step6_corrected_spectra")

## Checklist before trusting the result

* AGN crosses on the nucleus in every band (Section 1).
* Spaxel fits sensible for every line; weak / blended lines removed (Sections 2-3).
* Geometry: fixed to reliable values, or well constrained by the fits; models not mirrored (Section 4).
* Line groups chosen from the maps, not only from the IP guess (Section 4).
* Dispersion histograms show two components for every decomposed line (Section 3).
* Residual maps without coherent structure (Section 5).
* Stacked map stable when lines are left out or weights changed (Section 6).
* Corrected spectra (Section 8): curves of growth consistent with a point source (or `PSF_POINT_SOURCE = "fit"`);
  H$_2$ example diagrams smooth after correction (no ortho/para zig-zag); band scale factors close to 1.

The method assumes two kinematic components with $\sigma_{\rm outflow} > \sigma_{\rm disc}$, a rotating disc and a
(hollow) biconical outflow. For other geometries, replace a component with an imported map (Section 4) or extend
`breaking_the_blend.models`. Each step is a plain function, so individual lines can be re-done by hand and swapped into
`line_maps` / `decomps` before stacking (see the end of `01_NGC5728.ipynb`).